# Esercizio – Prevedere la parola successiva con una LSTM

## Obiettivo

Nell'esercizio precedente abbiamo utilizzato una `SimpleRNN` per
prevedere la parola successiva.

Ora ripetiamo lo stesso esercizio utilizzando una `LSTM`.

L'esercizio rimane volutamente molto simile a quello precedente.

La modifica principale è:

```text
SimpleRNN(100)  →  LSTM(100)
```

Il problema rimane lo stesso:

**una parola → parola successiva**

In [ ]:
# ============================================================
# 1. IMPORTIAMO LE LIBRERIE
# ============================================================

import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')

In [ ]:
# ============================================================
# 2. CARICAMENTO DEL TESTO
# ============================================================

# Carichiamo il testo dal file.
#
# lower() trasforma tutto il testo in minuscolo.

with open('testo_autore.txt', 'r', encoding='utf-8') as file:
    text = file.read().lower()

print("Prime 500 lettere del testo:")
print(text[:500])

In [ ]:
# ============================================================
# 3. TOKENIZZAZIONE
# ============================================================

# Dividiamo il testo in parole.

tokens = word_tokenize(text)

print("Prime 20 parole:")
print(tokens[:20])

print("\nNumero totale di parole:", len(tokens))

## 4. Creiamo il vocabolario

Utilizziamo `Tokenizer`, come nell'esercizio precedente.

Il Tokenizer assegna un numero a ogni parola.

### HINT

Completa le due righe mancanti.

In [ ]:
# ============================================================
# 4. CREAZIONE DEL VOCABOLARIO
# ============================================================

tokenizer = Tokenizer()

# HINT:
# fai analizzare al Tokenizer la lista delle parole.

tokenizer.____________________([tokens])

# HINT:
# calcola il numero totale di parole del vocabolario.
# Aggiungi 1 perché l'indice 0 è riservato.

total_words = ____________________

print("Numero totale di parole:", total_words)

## 5. Creiamo i bigrammi

Come nell'esercizio iniziale, creiamo coppie di parole consecutive.

Per esempio:

```text
il gatto dorme
```

diventa:

```text
il → gatto
gatto → dorme
```

Quindi la rete dovrà imparare a prevedere la parola successiva
partendo dalla parola precedente.

In [ ]:
# ============================================================
# 5. CREAZIONE DEI BIGRAMMI
# ============================================================

input_sequences = []

for i in range(1, len(tokens)):

    # Prendiamo due parole consecutive.

    n_gram_sequence = tokens[i-1:i+1]

    input_sequences.append(n_gram_sequence)


print("Prime 10 coppie:")

for seq in input_sequences[:10]:
    print(seq)

In [ ]:
# ============================================================
# 6. CREAZIONE DI X E y
# ============================================================

X = []
y = []

for seq in input_sequences:

    # Trasformiamo la parola precedente in un numero.

    prev_word = tokenizer.texts_to_sequences([seq[0]])

    # Trasformiamo la parola successiva in un numero.

    next_word = tokenizer.texts_to_sequences([seq[1]])

    # Controlliamo che le parole siano presenti
    # nel vocabolario.

    if (prev_word and next_word and
        len(prev_word[0]) > 0 and len(next_word[0]) > 0):

        X.append(prev_word[0][0])
        y.append(next_word[0][0])


X = np.array(X)
y = np.array(y)

print("Shape X prima del reshape:", X.shape)
print("Shape y:", y.shape)

In [ ]:
# ============================================================
# 7. PREPARIAMO X E y
# ============================================================

# Ogni input contiene una sola parola.
#
# Il modello deve quindi ricevere X con forma:
#
# (numero_esempi, 1)

X = X.reshape((X.shape[0], 1))

print("Shape X:", X.shape)

# Trasformiamo il target in one-hot encoding.

y = to_categorical(
    y,
    num_classes=total_words
)

print("Shape y:", y.shape)

## 8. Costruiamo il modello LSTM

Confronta questo modello con quello dell'esercizio RNN.

La struttura è:

```text
Embedding
    ↓
LSTM
    ↓
Dense
```

La sola modifica importante rispetto alla RNN è il livello ricorrente:

```python
SimpleRNN(100)
```

diventa:

```python
LSTM(100)
```

### HINT

Mantieni gli stessi parametri dell'esercizio precedente.

In [ ]:
# ============================================================
# 8. COSTRUZIONE DEL MODELLO
# ============================================================

model = Sequential()

# Embedding:
# total_words = numero di parole del vocabolario
# 50 = dimensione del vettore
# input_length=1 = una parola in ingresso

model.add(
    Embedding(
        ___,
        50,
        input_length=1
    )
)

# HINT:
# sostituisci la SimpleRNN dell'esercizio precedente
# con una LSTM mantenendo 100 neuroni.

model.add(
    LSTM(___)
)

# HINT:
# il Dense deve avere un'uscita per ogni parola
# del vocabolario.

model.add(
    Dense(
        ___,
        activation="softmax"
    )
)

model.summary()

In [ ]:
# ============================================================
# 9. COMPILIAMO IL MODELLO
# ============================================================

# Utilizziamo gli stessi parametri dell'esercizio precedente.

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
# ============================================================
# 10. ADDESTRIAMO LA RETE
# ============================================================

# Addestriamo il modello.
#
# Manteniamo lo stesso numero di epoche
# dell'esercizio precedente.

model.fit(
    X,
    y,
    epochs=100,
    verbose=1
)

## 11. Prevediamo la parola successiva

Ora utilizziamo il modello addestrato.

Partiamo da una parola, per esempio:

```text
il
```

e chiediamo alla rete quale parola potrebbe venire dopo.

La funzione seguente esegue questi passaggi:

1. trasforma la parola in un numero;
2. prepara l'input;
3. chiede la previsione al modello;
4. trova la probabilità maggiore;
5. cerca la parola corrispondente nel vocabolario.

In [ ]:
# ============================================================
# 11. FUNZIONE PER LA PREVISIONE
# ============================================================

def predict_next_word(seed_text, model, tokenizer):

    # Trasformiamo la parola in un numero.

    sequence = tokenizer.texts_to_sequences([seed_text])[0]

    # Se la parola non è presente nel vocabolario,
    # evitiamo un errore.

    if len(sequence) == 0:
        return "Parola non presente nel vocabolario"

    # Il modello si aspetta:
    # (numero_esempi, numero_timesteps)
    #
    # In questo esercizio:
    # un esempio e un timestep.

    sequence = np.array(sequence).reshape(1, 1)

    # Facciamo la previsione.

    prediction = model.predict(sequence, verbose=0)

    # Troviamo l'indice della probabilità maggiore.

    predicted_index = np.argmax(prediction[0])

    # Cerchiamo la parola corrispondente.

    for word, index in tokenizer.word_index.items():

        if index == predicted_index:
            return word

    return "Parola non trovata"

### 🔎 Attenzione agli indici

Il `Tokenizer` assegna alle parole gli indici a partire da `1`.

Nel nostro modello l'indice `0` non rappresenta una parola del vocabolario.

Per questo, se `argmax()` restituisce `0`, la funzione non trova
nessuna parola e restituisce `"Parola non trovata"`.

Non aggiungiamo `+1`: utilizziamo direttamente l'indice prodotto
dal modello.

In [ ]:
# ============================================================
# 12. PROVIAMO LA PREVISIONE
# ============================================================

seed_text = "il"

next_word = predict_next_word(
    seed_text,
    model,
    tokenizer
)

print("Parola iniziale:", seed_text)
print("Parola prevista:", next_word)

## 🎯 Prova con altre parole

Scegli una parola presente nel testo e modifica `seed_text`.

Osserva la parola prevista.

La previsione non deve necessariamente essere una parola
che segue sempre quella iniziale nel testo: la rete sta cercando
di imparare le regolarità presenti negli esempi di addestramento.

In [ ]:
# ============================================================
# 13. PROVA CON UN'ALTRA PAROLA
# ============================================================

# Inserisci una parola presente nel testo.

seed_text = "___"

next_word = predict_next_word(
    seed_text,
    model,
    tokenizer
)

print("Parola iniziale:", seed_text)
print("Parola prevista:", next_word)

# 📝 Domande finali

1. Qual è il compito della rete?

2. Che cosa contiene `X`?

3. Che cosa contiene `y`?

4. Perché utilizziamo l'Embedding?

5. Qual è la differenza principale tra questo esercizio
   e quello con la SimpleRNN?

6. Quale livello abbiamo cambiato?

7. Perché il Dense finale ha `total_words` neuroni?

8. Perché utilizziamo `softmax`?

9. Che cosa restituisce `model.predict()`?

10. Come troviamo la parola con probabilità maggiore?

11. Perché la previsione dipende dal testo utilizzato
    per l'addestramento?

## ⭐ Domanda finale

Nel codice abbiamo cambiato principalmente:

```text
SimpleRNN(100)
        ↓
LSTM(100)
```

Perché una LSTM può essere utile quando vogliamo
conservare informazioni per più passi temporali?